# Day 2: Analysis, machine learning and a Streamlit app
**Python for Data Science in Academic Research, with AI, VS Code & GitHub**
Nasarawa State University, Keffi · Saturday 31 October 2026 · 10:00 am – 2:00 pm

### By the end of today you will be able to
1. Test a hypothesis and report **effect sizes and confidence intervals**, not just p-values
2. Fit and interpret a **regression model** with statsmodels and check its assumptions
3. Build a **leak-free machine-learning pipeline** in scikit-learn and evaluate it honestly
4. Produce **publication-ready figures and tables**
5. Turn your model into an interactive **Streamlit app**
6. Write an **AI-use statement** for your thesis or paper

Symbols: 🤖 AI exercise · ✍️ your turn · 💾 Git checkpoint

### 💾 Start with a sync
```bash
git pull
```

In [ ]:
import sys
from pathlib import Path

# Make the project's src/ folder importable, whether the notebook runs from
# the repository root or from inside notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.plotting import set_publication_style, save_figure

set_publication_style()
pd.set_option("display.max_columns", 30)
print("Project root:", ROOT)

In [ ]:
from src.cleaning import load_processed, TARGET

df = load_processed()          # re-creates the clean file from raw data if it's missing
print(df.shape)
df.head()

## 1. Start from a research question
> **RQ:** Which farm practices are associated with higher maize yield among smallholder farmers in Nasarawa State?
>
> **H1:** Farms that use improved seed have a higher mean yield than farms that do not.

A clear question decides the method. *Explaining* (statistics) and *predicting* (machine learning) are different goals, and today we do both.

## 2. Hypothesis test: improved seed vs local seed

In [ ]:
from scipy import stats

improved = df.loc[df["improved_seed"] == "Yes", TARGET]
local = df.loc[df["improved_seed"] == "No", TARGET]

t, p = stats.ttest_ind(improved, local, equal_var=False)       # Welch's t-test

diff = improved.mean() - local.mean()
se = np.sqrt(improved.var() / len(improved) + local.var() / len(local))
ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se
pooled_sd = np.sqrt((improved.var() + local.var()) / 2)
cohens_d = diff / pooled_sd

print(f"Improved seed: n = {len(improved)}, mean = {improved.mean():,.0f} kg/ha")
print(f"Local seed:    n = {len(local)}, mean = {local.mean():,.0f} kg/ha")
print(f"Difference = {diff:,.0f} kg/ha (95% CI {ci_low:,.0f} to {ci_high:,.0f}), "
      f"Welch t = {t:.2f}, p = {p:.3g}, Cohen's d = {cohens_d:.2f}")

**How to report it (APA style):**
> Farms using improved seed had a higher mean yield than farms using local seed (difference = *X* kg/ha, 95% CI [*a*, *b*]), Welch's *t* = *…*, *p* < .001, Cohen's *d* = *…*.

Reviewers want to know **how big** the effect is (effect size and CI), not only whether *p* < 0.05.

⚠️ This is an *association*. Farmers who buy improved seed may also differ in other ways, such as fertiliser use or cooperative membership. Regression helps us adjust for that.

## 3. Multiple regression with statsmodels

In [ ]:
import statsmodels.formula.api as smf

formula = (
    "yield_kg_per_ha ~ fertilizer_kg_per_ha + rainfall_mm + soil_ph + extension_visits"
    " + C(improved_seed, Treatment('No')) + C(irrigation, Treatment('No'))"
    " + C(pest_incidence, Treatment('Absent')) + C(education_level, Treatment('No formal education'))"
)
ols = smf.ols(formula, data=df).fit()
print(f"Rows used: {int(ols.nobs)} of {len(df)} (rows with missing values are dropped, so report this!)")
ols.summary()

### A tidy coefficient table for your thesis

In [ ]:
coef_table = pd.DataFrame({
    "coef": ols.params,
    "ci_low": ols.conf_int()[0],
    "ci_high": ols.conf_int()[1],
    "p_value": ols.pvalues,
}).round(3)

RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)
coef_table.to_csv(RESULTS_DIR / "table2_regression_coefficients.csv")
coef_table

### Check the assumptions before you believe the results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].scatter(ols.fittedvalues, ols.resid, alpha=0.4)
axes[0].axhline(0, color="black", lw=1)
axes[0].set(xlabel="Fitted values", ylabel="Residuals", title="Residuals vs fitted")
stats.probplot(ols.resid, dist="norm", plot=axes[1])
axes[1].set_title("Normal Q-Q plot of residuals")
plt.tight_layout()

Look for a **curve** in *residuals vs fitted*: it suggests a relationship that isn't a straight line. (Hint: is yield really a straight-line function of rainfall?)

### 🤖 AI exercise 3: interpret, then verify
Paste the coefficient table (not the data) into your AI assistant:
> *"Interpret this regression table for the results chapter of an MSc thesis in plain academic English. The outcome is maize yield in kg/ha. Mention the direction, size and confidence interval of each statistically significant effect, and state the limitations of observational data."*

✅ **Check every number** in the AI's paragraph against the table above. AI tools often get signs or units wrong and overstate causality.

### ✍️ Your turn
Add a squared rainfall term, `+ I((rainfall_mm - 1150)**2)`, to the formula. Does adjusted R² improve? Does the residual plot look better?

In [ ]:
# ✍️ Your improved model here

## 4. Machine learning: can we *predict* yield for a new farm?
### 4.1 Hold out a test set first
The test set is locked away until the very end. Every decision (model choice, settings) uses the training set only.

In [ ]:
from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

from src.modelling import (FEATURES, NUMERIC_FEATURES, CATEGORICAL_FEATURES, RANDOM_STATE,
                           build_preprocessor, regression_metrics, save_model)

X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train: {len(X_train)} farms   Test: {len(X_test)} farms")
print("Numeric features:    ", NUMERIC_FEATURES)
print("Categorical features:", CATEGORICAL_FEATURES)

### 4.2 A *pipeline* keeps preprocessing inside cross-validation
`build_preprocessor()` (in `src/modelling.py`) fills missing values, scales the numbers and one-hot encodes the categories. Because it sits **inside** the pipeline, those steps are re-learned on each training fold, so there is no leakage.

In [ ]:
candidates = {
    "Baseline (predict the mean)": DummyRegressor(),
    "Linear regression": LinearRegression(),
    "Random forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient boosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
}
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, model in candidates.items():
    pipe = Pipeline([("preprocess", build_preprocessor()), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv,
                            scoring=["r2", "neg_mean_absolute_error"])
    rows.append({"model": name,
                 "R2 (mean)": scores["test_r2"].mean(), "R2 (sd)": scores["test_r2"].std(),
                 "MAE (mean)": -scores["test_neg_mean_absolute_error"].mean()})

cv_results = pd.DataFrame(rows).set_index("model").round(3)
cv_results.to_csv(RESULTS_DIR / "table3_model_comparison_cv.csv")
cv_results

Always include a **baseline**. A model is only useful if it clearly beats "just predict the average".

### 4.3 ⚠️ The leakage trap
A colleague says: *"Add `harvest_sales_naira` divided by farm size, called sales per hectare. It makes the model much better!"* Let's see:

In [ ]:
leaky = X_train.copy()
leaky["sales_per_ha"] = df.loc[X_train.index, "harvest_sales_naira"] / df.loc[X_train.index, "farm_size_ha"]

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

leaky_pipe = Pipeline([
    ("preprocess", ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), NUMERIC_FEATURES + ["sales_per_ha"]),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                          ("encode", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL_FEATURES),
    ])),
    ("model", RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)),
])
leaky_r2 = cross_val_score(leaky_pipe, leaky, y_train, cv=cv, scoring="r2").mean()
print(f"R² WITH the leaky feature: {leaky_r2:.3f}   (looks amazing...)")
print(f"R² without it:             {cv_results.loc['Random forest', 'R2 (mean)']:.3f}")

Sales are only known **after** harvest, and they are calculated from yield (sales = yield × area × price). The "amazing" model is useless for predicting a farm's yield in advance. **If a result looks too good to be true, check for leakage.** Common sources in research data are post-outcome variables, duplicate subjects across train and test, and scaling or imputing before splitting.

### 4.4 Final evaluation on the untouched test set

In [ ]:
best_name = cv_results["R2 (mean)"].idxmax()
final_pipeline = Pipeline([("preprocess", build_preprocessor()), ("model", candidates[best_name])])
final_pipeline.fit(X_train, y_train)

y_pred = final_pipeline.predict(X_test)
test_metrics = regression_metrics(y_test, y_pred)
print(f"Chosen model: {best_name}")
print({k: round(v, 3) for k, v in test_metrics.items()})

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_test, y_pred, alpha=0.6)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, color="black", lw=1, ls="--", label="Perfect prediction")
ax.set(xlabel="Actual yield (kg/ha)", ylabel="Predicted yield (kg/ha)",
       title=f"{best_name}: test R² = {test_metrics['R2']:.2f}")
ax.legend()
save_figure(fig, "fig3_predicted_vs_actual")
plt.show()

### 4.5 Which features matter? (permutation importance)

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(final_pipeline, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
importance = (pd.DataFrame({"feature": FEATURES, "importance": perm.importances_mean, "sd": perm.importances_std})
              .sort_values("importance"))

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.barh(importance["feature"], importance["importance"], xerr=importance["sd"], color="#0072B2")
ax.set(xlabel="Drop in R² when the feature is shuffled", title="Permutation feature importance (test set)")
save_figure(fig, "fig4_feature_importance")
plt.show()

⚠️ *Important to the model* does **not** mean *causes yield*. Importance describes the model, not the world.

## 5. A publication-ready figure
A **coefficient (forest) plot** is often clearer than a long regression table. The code below uses the style from `src/plotting.py`: 300 dpi, a colour-blind-safe palette, labelled axes with units, and PNG for Word plus PDF for LaTeX.

In [ ]:
plot_df = coef_table.drop(index="Intercept").copy()
plot_df.index = (plot_df.index
                 .str.replace(r"C\((\w+), Treatment\('\w+'\)\)\[T\.(.+)\]", r"\1 = \2", regex=True)
                 .str.replace("_", " "))
plot_df = plot_df.sort_values("coef")

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.errorbar(plot_df["coef"], plot_df.index, xerr=[plot_df["coef"] - plot_df["ci_low"], plot_df["ci_high"] - plot_df["coef"]],
            fmt="o", color="#0072B2", ecolor="#56B4E9", capsize=3)
ax.axvline(0, color="black", lw=1)
ax.set(xlabel="Change in yield (kg/ha), 95% CI", title="Factors associated with maize yield")
save_figure(fig, "fig2_regression_coefficients")
plt.show()

> Note: continuous variables such as rainfall are "per 1 mm", so their bars look tiny. For your paper, consider rescaling (e.g. per 100 mm) so the effects can be compared.

## 6. Save the model and its model card

In [ ]:
save_model(final_pipeline, test_metrics, X_train, notes=f"Chosen by 5-fold CV in notebook 02 ({best_name})")
print((ROOT / "models" / "model_card.json").read_text()[:600], "...")

## 7. 🚀 Demo your model with Streamlit
In the VS Code terminal, from the **repository root**, run:
```bash
streamlit run app/streamlit_app.py
```
Your browser opens the app. Try it:
- **Predict one farm**: change the inputs and see the predicted yield and the fertiliser "what if" curve
- **Predict from a CSV file**: upload `data/raw/nasarawa_maize_survey_synthetic.csv`
- **About the model**: the model card, so users know what the model can and cannot do

Open `app/streamlit_app.py`. It is about 120 lines of plain Python, and every `st.something(...)` call adds one element to the page.

### 🤖 AI exercise 4: extend the app
Ask your AI assistant (or Copilot Chat in VS Code, with the file open):
> *"In this Streamlit app, add a new tab called 'Explore data' that shows a bar chart of the average yield by LGA from `get_training_data()`, with a selectbox to choose between mean and median."*

Save the file. Streamlit reloads automatically. Does it work? Do you understand the code it added?

**Share it (optional):** push the repo to GitHub, then deploy for free at [share.streamlit.io](https://share.streamlit.io) (Community Cloud) and paste the link in your README.

## 8. ✍️ Mini-project: your own data
1. Put your dataset in `data/raw/` (non-confidential only, or add its file name to `.gitignore`)
2. Copy `01_day1_clean_and_explore.ipynb` and adapt the audit and cleaning for your data
3. State **one research question** and answer it with a test or regression **and** a figure
4. *Stretch:* train a model, save it with `save_model(...)`, update the feature lists in `src/modelling.py` and run the app
5. Update `README.md`: question, data source, how to run, key finding

## 9. Responsible AI and finishing your write-up
- Fill in `docs/AI_USE_STATEMENT_TEMPLATE.md` and keep it with your thesis
- Work through `docs/REPRODUCIBILITY_CHECKLIST.md`
- **Never** let AI write citations for you without checking each one. AI tools invent references that look real.

### 💾 Final commit
```bash
git add .
git commit -m "Day 2: analysis, model and Streamlit app"
git push
```
🎓 **Optional:** connect the repository to [Zenodo](https://zenodo.org) and make a GitHub *release* to get a citable **DOI** for your code.